In [1]:
import json
import os
import time
from PIL import Image
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import jax
import jax.numpy as jnp
from flax.training import checkpoints
import numpy as np

from experiments.mappings import NEW_MAPPING
from examples.utils import read_utils

from serl_launcher.agents.continuous.sac import SACAgent
from gymnasium.wrappers.record_episode_statistics import RecordEpisodeStatistics

from agentlace.data.data_store import QueuedDataStore
from serl_launcher.utils.launcher import make_sac_pixel_agent

from transforms3d.quaternions import quat2mat
import importlib
import cv2


importlib.reload(read_utils)

devices = jax.local_devices()
num_devices = len(devices)
sharding = jax.sharding.PositionalSharding(devices)

base_path = '/home/qiangqiang/workspaces/data/test_data/ball_pick_2025_02_12_00'
json_path = os.path.expanduser('/home/qiangqiang/workspaces/data/test_data/ball_pick_2025_02_12_00/clip_marks.json')
robot_urdf_path = "/home/qiangqiang/workspaces/HK_TACTEXO_DATA/denso_robot_with_ati_4.urdf"


checkpoint_path = "experiments/tennis_ball_pick/second_run"
exp_name = "tennis_ball_pick"
eval_checkpoint_step = 5000
seed = 42
learner = False
save_video = False

cam_front_translation = [1.2367936975704506, 0.032497565951025945, 0.5359742126690214]
cam_front_quaternion = [0.012480230443529135, 0.27804828390806924, -0.026321127948753298, -0.960125301139054] # [w, x, y, z]
# Convert quaternion to rotation matrix
cam_front_rotation_matrix = quat2mat([cam_front_quaternion[0], cam_front_quaternion[1],
                                      cam_front_quaternion[2], cam_front_quaternion[3]])

camfront2robot = np.eye(4)
camfront2robot[:3, :3] = cam_front_rotation_matrix
camfront2robot[:3, 3] = cam_front_translation
T = np.array([
    [0, 0, 1, 0], # Maps z -> x
    [-1, 0, 0, 0], # Maps -x -> y
    [0, -1, 0, 0], # Maps -y -> z
    [0, 0, 0, 1],  # Homogeneous coordinate unchanged
 ])

camfront2robot = camfront2robot @ T

robot2camfront = np.linalg.inv(camfront2robot)
fx = 385.732666015625
fy = 385.2535705566406
cx = 324.3997497558594
cy = 240.08477783203125



trajectory_points = []

def print_green(x):
    return print("\033[92m {}\033[00m".format(x))

def camera_to_pixel(camera_coords, K):
    """
    将相机坐标系中的点转换到像素坐标系
    :param camera_coords: 相机坐标系中的点，形状为 (3,) 或 (N, 3)
    :param K: 相机内参矩阵，形状为 (3, 3)
    :return: 像素坐标系中的点，形状为 (2,) 或 (N, 2)
    """
    if camera_coords.ndim == 1:
        camera_coords = camera_coords.reshape(1, 3)  # 转换为 (1, 3)

    # 投影到图像坐标系
    pixel_coords_homogeneous = K @ camera_coords.T  # (3, N)
    pixel_coords = pixel_coords_homogeneous[:2] / pixel_coords_homogeneous[2]  # 归一化

    if pixel_coords.shape[1] == 1:
        return pixel_coords.flatten()  # 返回 (2,)
    else:
        return pixel_coords.T  # 返回 (N, 2)

def actor_test(agent, env, sampling_rng):
    time_list = []

    ckpt = checkpoints.restore_checkpoint(
        os.path.abspath(checkpoint_path),
        agent.state,
        step=eval_checkpoint_step,
    )
    print_green(f"Loaded previous checkpoint at step {eval_checkpoint_step}.")

    # print("agent before = ", agent)
    agent = agent.replace(state=ckpt)
    # print("agent after = ", agent)

    data = read_utils.read_data(robot_urdf_path, True)

    # obs, _ = env.reset()
    log_file = "classifier_log_in_training.txt"
    global trajectory_points
    trajectory_points = []
    trajectory_colors = []

    try:
        with open(log_file, "w") as f:
            count = 0
            count_pixel = 0
            for data_count in range(len(data)):
                env.unwrapped.set_data_count(data_count)
                obs = data[data_count]["observations"]

                sampling_rng, key = jax.random.split(sampling_rng)
                # actions = data[data_count]["observations"]["state"]
                print("obs shape= ", data[data_count]["observations"]["front_camera"].shape)
                print("obs shape= ", data[data_count]["observations"]["side_camera"].shape)
                print("obs shape= ", data[data_count]["observations"]["state"].shape)
                print("obs state= ", data[data_count]["observations"]["state"])
                actions = agent.sample_actions(
                    observations=jax.device_put(obs),
                    argmax=False,
                    seed=key
                )
                actions = np.asarray(jax.device_get(actions))
                print("action shape = ", actions.shape)
                print("action  = ", actions)
                # print("actions = ", actions)
                # 将机械臂动作转换到相机坐标系
                action_in_robot_frame = np.array([actions[0], actions[1], actions[2], 1])
                # print("action_in_robot_frame = ", action_in_robot_frame)
                if count == 0:
                    log_msg = f"obs = {obs}\naction_in_robot_frame = {action_in_robot_frame}\n"
                    # print(log_msg)  # 仍然可以在终端输出
                    f.write(log_msg)  # 同时写入日志文件
                    count+=1

                # if count%3==0:
                #     action_in_robot_frame = np.array([0, 0, 0, 1])
                #     action_in_robot_frame = np.array([0, 0 + count*0.01, 0, 1])
                # elif count%3==1:
                #     action_in_robot_frame = np.array([0.74301654, -0.06079092, 0.26904497, 1])
                #     action_in_robot_frame = np.array([0+count*0.01, 0, 0, 1])

                # else:
                #     action_in_robot_frame = np.array([0.74301654, -0.07079092, 0.26904497, 1])
                #     action_in_robot_frame = np.array([0, 0, 0+count*0.01, 1])

                action_in_cam_frame =  robot2camfront @ action_in_robot_frame
                # 将相机坐标系中的点转换到像素坐标系
                K = np.array([
                    [fx, 0, cx],
                    [0, fy, cy],
                    [0, 0, 1]
                ])
                camera_coords = action_in_cam_frame[:3]  # 取前 3 个元素 (X_c, Y_c, Z_c)
                # print("camera_coords = ", camera_coords)
                pixel_coords = camera_to_pixel(camera_coords, K)  # 像素坐标系中的点 (u, v)
                # if count_pixel == 0:
                #     print("pixel_coords = ", pixel_coords)
                #     count_pixel = 1


                # 将动作点添加到轨迹中
                # print("pixel_coords = ", pixel_coords)
                trajectory_points.append(pixel_coords)
                trajectory_colors.append('red')
                f.flush()

    except KeyboardInterrupt:
        print("\nUser interrupted the program. Printing final logs...")
    
    finally:
        # print(f"success rate: {success_counter / eval_n_trajs}")
        print(f"average time: {np.mean(time_list)}")
    
     # 显示图像和轨迹的函数
    def display_image_with_trajectory(index):
        original_width, original_height = 640, 480
        target_width, target_height = 640, 480
        scale_x = target_width / original_width
        scale_y = target_height / original_height
        # 加载图像
        image = data[index]["observations"]["front_camera"]
        resized_image = cv2.resize(image, (target_width, target_height))

        # 显示图像
        plt.imshow(resized_image)

        # 绘制轨迹点
        if trajectory_points:
            trajectory_points_array = np.array(trajectory_points)
            trajectory_colors_array = np.array(trajectory_colors)
            # 过滤超出图像尺寸的点
            valid_points = []
            valid_colors = []
            for i, point in enumerate(trajectory_points_array):
                u, v = point
                new_u, new_v = int(u * scale_x), int(v * scale_y)
                # print("point = ", point)
                if 0 <= new_u < target_width*2 and 0 <= new_v < target_height*2:  # 图像尺寸为 640x480
                    valid_points.append((new_u, new_v))
                    valid_colors.append(trajectory_colors_array[i])
            if valid_points:
                valid_points = np.array(valid_points)
                valid_colors = np.array(valid_colors, dtype='<U10')
                if index < len(valid_points):
                    valid_colors[index] = 'blue'
                # valid_colors = [str(color) for color in valid_colors]
                print(f"Valid trajectory points: {len(valid_points)}/{len(trajectory_points_array)}")
                plt.scatter(valid_points[:, 0], valid_points[:, 1], c=valid_colors, s=10, label='Trajectory')

        plt.title(f"Frame: {index}")
        plt.axis('off')
        plt.legend()
        plt.show()

    interact(display_image_with_trajectory, index=IntSlider(min=0, max=len(data)-1, step=1, value=0))
        

def main():
    global config
    config = NEW_MAPPING[exp_name]()

    assert config.batch_size % num_devices == 0
    # seed
    rng = jax.random.PRNGKey(seed)
    rng, sampling_rng = jax.random.split(rng)

    assert exp_name in NEW_MAPPING, "Experiment folder not found."
    env = config.get_environment(
        fake_env=learner,
        save_video=save_video,
        classifier=True,
    )
    env = RecordEpisodeStatistics(env)

    rng, sampling_rng = jax.random.split(rng)

    agent: SACAgent = make_sac_pixel_agent(
        seed=seed,
        sample_obs=env.observation_space.sample(),
        sample_action=env.action_space.sample(),
        image_keys=config.image_keys,
        encoder_type=config.encoder_type,
        discount=config.discount,
    )

    # replicate agent across devices
    # need the jnp.array to avoid a bug where device_put doesn't recognize primitives
    agent = jax.device_put(
        jax.tree_map(jnp.array, agent), sharding.replicate()
    )

    if checkpoint_path is not None and os.path.exists(checkpoint_path):
        input("Checkpoint path already exists. Press Enter to resume training.")
        ckpt = checkpoints.restore_checkpoint(
            os.path.abspath(checkpoint_path),
            agent.state,
        )
        agent = agent.replace(state=ckpt)
        # ckpt_number = os.path.basename(
        #     checkpoints.latest_checkpoint(os.path.abspath(checkpoint_path))
        # )[11:]
        # print_green(f"Loaded previous checkpoint at step {ckpt_number}.")
    
    sampling_rng = jax.device_put(sampling_rng, sharding.replicate())

    # actor loop
    print_green("starting actor loop")
    actor_test(
        agent,
        env,
        sampling_rng,
    )


if __name__ == "__main__":
    main()


2025-03-19 18:54:09.969051: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742378049.982978  686432 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742378049.987229  686432 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Initialized Denso
The ResNet-10 weights already exist at '/home/qiangqiang/.serl/resnet10_params.pkl'.
Loaded 5.418792M parameters from ResNet-10 pretrained on ImageNet-1K
replaced conv_init in encoder_front_camera
replaced norm_init in encoder_front_camera
replaced ResNetBlock_0 in encoder_front_camera
replaced ResNetBlock_1 in encoder_front_camera
replaced ResNetBlock_2 in encoder_front_camera
replaced ResNetBlock_3 in encoder_front_camera
The ResNet-10 weights already exist at '/home/qiangqiang/.serl/resnet10_params.pkl'.
Loaded 5.418792M parameters from ResNet-10 pretrained on ImageNet-1K
replaced conv_init in pretrained_encoder
replaced norm_init in pretrained_encoder
replaced ResNetBlock_0 in pretrained_encoder
replaced ResNetBlock_1 in pretrained_encoder
replaced ResNetBlock_2 in pretrained_encoder
replaced ResNetBlock_3 in pretrained_encoder


/tmp/ipykernel_686432/2772097385.py:262: DeprecationWarning: jax.tree_map is deprecated: use jax.tree.map (jax v0.4.25 or newer) or jax.tree_util.tree_map (any JAX version).
  jax.tree_map(jnp.array, agent), sharding.replicate()


 starting actor loop


ValueError: Matching checkpoint not found: /home/qiangqiang/workspaces/hil-serl/examples/experiments/tennis_ball_pick/second_run/checkpoint_5000